# Clase 19 — Caso de Estudio: *SaludDirecta S.L.*

Pipeline de **limpieza y análisis** del registro de pacientes y tratamientos usando funciones integradas de Spark y una UDF.

## 🏢 Contexto

SaludDirecta S.L. es una red de clínicas privadas. Tras años acumulando datos exportados desde tres sistemas distintos, el fichero de pacientes presenta inconsistencias graves: nombres con mayúsculas/minúsculas sin criterio, teléfonos con formatos dispares, fechas como texto plano, y el historial de tratamientos almacenado como listas separadas por `;`.

Se construye un pipeline en 7 tareas para preparar los datos antes de migrarlos al nuevo sistema.


## Celda 1 — Inicializar SparkSession

In [4]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`
import org.apache.log4j.{Level, Logger}
Logger.getLogger("org").setLevel(Level.ERROR)
Logger.getLogger("akka").setLevel(Level.ERROR)

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._

val spark = SparkSession.builder()
  .appName("Clase17-Sesion1-DataFrames")
  .master("local[*]")
  .config("spark.ui.showConsoleProgress", "false")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
import spark.implicits._

println(s"✅ Spark ${spark.version} listo")

✅ Spark 4.1.1 listo


import $ivy.$
import org.apache.log4j.{Level, Logger}
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@1e6bef54
import spark.implicits._

## Celda 2 — Datos de pacientes (registro con datos sucios)

In [5]:
val pacientesRaw = Seq(
  (1,  "  ANA  garcía torres  ", "ana.garcia@mail.ES",    "612-345 678", "1985-07-14", "2019-03-01", 2),
  (2,  "PEDRO López",            "pedro.lopez@MAIL.com",  "699 876 543", "1972-11-22", "2021-06-15", 5),
  (3,  "maría RUIZ",             "maria.ruiz@mail.com",   "654321098",   "1990-04-05", "2023-01-10", 1),
  (4,  "  Carlos SANZ  ",        "CARLOS.SANZ@mail.es",   "600-111-222", "1968-09-30", "2020-08-22", 4),
  (5,  "laura vega",              "laura.vega@mail.com",  "677 999 000", "2001-02-18", "2022-12-01", 3),
  (6,  "JORGE martínez",         "jorge.martinez@MAIL.ES","655 432-109", "1955-12-03", "2018-05-17", 7),
  (7,  "  Sofía RAMOS  ",        "sofia.ramos@mail.com",  "610-222 333", "1995-08-25", "2024-02-28", 2),
  (8,  "ANTONIO Vidal",          "antonio.vidal@mail.ES", "699-000 111", "1980-01-09", "2017-11-05", 9)
).toDF("id", "nombre", "email", "telefono", "fecha_nacimiento", "fecha_alta", "num_visitas")

pacientesRaw.show(truncate = false)

+---+----------------------+----------------------+-----------+----------------+----------+-----------+
|id |nombre                |email                 |telefono   |fecha_nacimiento|fecha_alta|num_visitas|
+---+----------------------+----------------------+-----------+----------------+----------+-----------+
|1  |  ANA  garcía torres  |ana.garcia@mail.ES    |612-345 678|1985-07-14      |2019-03-01|2          |
|2  |PEDRO López           |pedro.lopez@MAIL.com  |699 876 543|1972-11-22      |2021-06-15|5          |
|3  |maría RUIZ            |maria.ruiz@mail.com   |654321098  |1990-04-05      |2023-01-10|1          |
|4  |  Carlos SANZ         |CARLOS.SANZ@mail.es   |600-111-222|1968-09-30      |2020-08-22|4          |
|5  |laura vega            |laura.vega@mail.com   |677 999 000|2001-02-18      |2022-12-01|3          |
|6  |JORGE martínez        |jorge.martinez@MAIL.ES|655 432-109|1955-12-03      |2018-05-17|7          |
|7  |  Sofía RAMOS         |sofia.ramos@mail.com  |610-222 333|1

pacientesRaw: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string ... 5 more fields]

## Celda 3 — Historial de tratamientos (texto con separadores)

In [6]:
// Tratamientos almacenados como texto separado por ";"
// El paciente 7 no tiene tratamientos registrados aún
val tratamientosRaw = Seq(
  (1, "fisioterapia;radiología;análisis"),
  (2, "cardiología;endocrinología;fisioterapia;cardiología"),
  (3, "análisis"),
  (4, "traumatología;fisioterapia;traumatología;radiología"),
  (5, "dermatología;análisis;dermatología"),
  (6, "cardiología;radiología;neurología;cardiología;fisioterapia"),
  (7, ""),
  (8, "traumatología;neurología;cardiología;traumatología;neurología")
).toDF("id", "tratamientos_texto")

tratamientosRaw.show(truncate = false)

+---+-------------------------------------------------------------+
|id |tratamientos_texto                                           |
+---+-------------------------------------------------------------+
|1  |fisioterapia;radiología;análisis                             |
|2  |cardiología;endocrinología;fisioterapia;cardiología          |
|3  |análisis                                                     |
|4  |traumatología;fisioterapia;traumatología;radiología          |
|5  |dermatología;análisis;dermatología                           |
|6  |cardiología;radiología;neurología;cardiología;fisioterapia   |
|7  |                                                             |
|8  |traumatología;neurología;cardiología;traumatología;neurología|
+---+-------------------------------------------------------------+



tratamientosRaw: org.apache.spark.sql.package.DataFrame = [id: int, tratamientos_texto: string]

---

## 🔧 Tarea 1 — Limpiar el DataFrame de pacientes

- **`nombre`:** `trim` + `initcap` (con normalización de espacios internos).
- **`email`:** `trim` + `lower`.
- **`telefono`:** eliminar todo lo que no sea dígito (`regexp_replace`).
- **`fecha_nacimiento`** y **`fecha_alta`:** `to_date` con patrón `"yyyy-MM-dd"`.


In [7]:
val pacientesLimpios = pacientesRaw
  .withColumn("nombre",   initcap(regexp_replace(trim($"nombre"), "\\s+", " ")))
  .withColumn("email",    lower(trim($"email")))
  .withColumn("telefono", regexp_replace($"telefono", "[^0-9]", ""))
  .withColumn("fecha_nacimiento", to_date($"fecha_nacimiento", "yyyy-MM-dd"))
  .withColumn("fecha_alta",       to_date($"fecha_alta",       "yyyy-MM-dd"))

pacientesLimpios.show(truncate = false)
pacientesLimpios.printSchema()

+---+-----------------+----------------------+---------+----------------+----------+-----------+
|id |nombre           |email                 |telefono |fecha_nacimiento|fecha_alta|num_visitas|
+---+-----------------+----------------------+---------+----------------+----------+-----------+
|1  |Ana García Torres|ana.garcia@mail.es    |612345678|1985-07-14      |2019-03-01|2          |
|2  |Pedro López      |pedro.lopez@mail.com  |699876543|1972-11-22      |2021-06-15|5          |
|3  |María Ruiz       |maria.ruiz@mail.com   |654321098|1990-04-05      |2023-01-10|1          |
|4  |Carlos Sanz      |carlos.sanz@mail.es   |600111222|1968-09-30      |2020-08-22|4          |
|5  |Laura Vega       |laura.vega@mail.com   |677999000|2001-02-18      |2022-12-01|3          |
|6  |Jorge Martínez   |jorge.martinez@mail.es|655432109|1955-12-03      |2018-05-17|7          |
|7  |Sofía Ramos      |sofia.ramos@mail.com  |610222333|1995-08-25      |2024-02-28|2          |
|8  |Antonio Vidal    |antonio

pacientesLimpios: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string ... 5 more fields]

---

## 📅 Tarea 2 — Enriquecer con columnas derivadas de fecha

- **`edad`:** años desde `fecha_nacimiento` hasta hoy.
- **`anios_como_paciente`:** años desde `fecha_alta` hasta hoy.
- **`fecha_alta_formato_es`:** `dd/MM/yyyy`.
- **`decada_nacimiento`:** década (1950, 1960, ...).


In [8]:
val pacientesEnriquecidos = pacientesLimpios
  .withColumn("edad",                 (months_between(current_date(), $"fecha_nacimiento") / 12).cast("int"))
  .withColumn("anios_como_paciente",  (months_between(current_date(), $"fecha_alta")       / 12).cast("int"))
  .withColumn("fecha_alta_formato_es", date_format($"fecha_alta", "dd/MM/yyyy"))
  .withColumn("decada_nacimiento",    ((year($"fecha_nacimiento") / 10).cast("int") * 10))

pacientesEnriquecidos
  .select("id", "nombre", "edad", "anios_como_paciente", "fecha_alta_formato_es", "decada_nacimiento")
  .show(truncate = false)

+---+-----------------+----+-------------------+---------------------+-----------------+
|id |nombre           |edad|anios_como_paciente|fecha_alta_formato_es|decada_nacimiento|
+---+-----------------+----+-------------------+---------------------+-----------------+
|1  |Ana García Torres|40  |7                  |01/03/2019           |1980             |
|2  |Pedro López      |53  |4                  |15/06/2021           |1970             |
|3  |María Ruiz       |36  |3                  |10/01/2023           |1990             |
|4  |Carlos Sanz      |57  |5                  |22/08/2020           |1960             |
|5  |Laura Vega       |25  |3                  |01/12/2022           |2000             |
|6  |Jorge Martínez   |70  |7                  |17/05/2018           |1950             |
|7  |Sofía Ramos      |30  |2                  |28/02/2024           |1990             |
|8  |Antonio Vidal    |46  |8                  |05/11/2017           |1980             |
+---+----------------

pacientesEnriquecidos: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string ... 9 more fields]

---

## 🔧 Tarea 3 — Clasificar pacientes con `when/otherwise`

| Condición | Valor |
| --- | --- |
| `num_visitas >= 7` | `Frecuente` |
| `num_visitas >= 4` | `Habitual` |
| `num_visitas >= 2` | `Ocasional` |
| resto | `Nuevo` |


In [9]:
val pacientesPerfil = pacientesEnriquecidos
  .withColumn("perfil_visitas",
    when($"num_visitas" >= 7, "Frecuente")
      .when($"num_visitas" >= 4, "Habitual")
      .when($"num_visitas" >= 2, "Ocasional")
      .otherwise("Nuevo"))

pacientesPerfil
  .select("id", "nombre", "num_visitas", "perfil_visitas")
  .show(truncate = false)

+---+-----------------+-----------+--------------+
|id |nombre           |num_visitas|perfil_visitas|
+---+-----------------+-----------+--------------+
|1  |Ana García Torres|2          |Ocasional     |
|2  |Pedro López      |5          |Habitual      |
|3  |María Ruiz       |1          |Nuevo         |
|4  |Carlos Sanz      |4          |Habitual      |
|5  |Laura Vega       |3          |Ocasional     |
|6  |Jorge Martínez   |7          |Frecuente     |
|7  |Sofía Ramos      |2          |Ocasional     |
|8  |Antonio Vidal    |9          |Frecuente     |
+---+-----------------+-----------+--------------+



pacientesPerfil: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string ... 10 more fields]

---

## 🔧 Tarea 4 — Preparar el historial de tratamientos

**4a.** `split` para convertir el texto en `ArrayType`.
**4b.** Filtrar pacientes sin tratamientos antes de hacer split.
**4c.** Añadir `num_tratamientos_totales`, `tratamientos_unicos`, `num_especialidades`.


In [10]:
// 4a + 4b: filtrar antes del split y luego dividir
val tratamientosArray = tratamientosRaw
  .filter($"tratamientos_texto" =!= "")
  .withColumn("tratamientos_array", split($"tratamientos_texto", ";"))

// 4c: métricas sobre el array
val tratamientosAnalisis = tratamientosArray
  .withColumn("num_tratamientos_totales", size($"tratamientos_array"))
  .withColumn("tratamientos_unicos",      array_distinct($"tratamientos_array"))
  .withColumn("num_especialidades",       size($"tratamientos_unicos"))

tratamientosAnalisis.show(truncate = false)

+---+-------------------------------------------------------------+-------------------------------------------------------------------+------------------------+---------------------------------------------------+------------------+
|id |tratamientos_texto                                           |tratamientos_array                                                 |num_tratamientos_totales|tratamientos_unicos                                |num_especialidades|
+---+-------------------------------------------------------------+-------------------------------------------------------------------+------------------------+---------------------------------------------------+------------------+
|1  |fisioterapia;radiología;análisis                             |[fisioterapia, radiología, análisis]                               |3                       |[fisioterapia, radiología, análisis]               |3                 |
|2  |cardiología;endocrinología;fisioterapia;cardiología          |[card

tratamientosArray: org.apache.spark.sql.package.DataFrame = [id: int, tratamientos_texto: string ... 1 more field]
tratamientosAnalisis: org.apache.spark.sql.package.DataFrame = [id: int, tratamientos_texto: string ... 4 more fields]

---

## 🔧 Tarea 5 — UDF `categorizarCasoUDF`

| Condición | Categoría |
| --- | --- |
| `null` | `Sin datos` |
| `>= 4` | `Caso complejo` |
| `>= 2` | `Caso moderado` |
| resto | `Caso simple` |


In [11]:
import org.apache.spark.sql.functions.udf

val categorizarCasoUDF = udf { (n: java.lang.Integer) =>
  if (n == null)        "Sin datos"
  else if (n >= 4)      "Caso complejo"
  else if (n >= 2)      "Caso moderado"
  else                  "Caso simple"
}

// Registro para usar también en Spark SQL (Tarea 7)
spark.udf.register("categorizar_caso", categorizarCasoUDF)

val tratamientosClasificados = tratamientosAnalisis
  .withColumn("complejidad_caso", categorizarCasoUDF($"num_especialidades"))

tratamientosClasificados
  .select("id", "num_especialidades", "complejidad_caso")
  .show(truncate = false)

+---+------------------+----------------+
|id |num_especialidades|complejidad_caso|
+---+------------------+----------------+
|1  |3                 |Caso moderado   |
|2  |3                 |Caso moderado   |
|3  |1                 |Caso simple     |
|4  |3                 |Caso moderado   |
|5  |2                 |Caso moderado   |
|6  |4                 |Caso complejo   |
|8  |3                 |Caso moderado   |
+---+------------------+----------------+



import org.apache.spark.sql.functions.udf
categorizarCasoUDF: org.apache.spark.sql.expressions.UserDefinedFunction = SparkUserDefinedFunction(
  f = ammonite.$sess.cmd11$Helper$$Lambda$5876/0x0000024982508000@f0d48c1,
  dataType = StringType,
  inputEncoders = ArraySeq(Some(value = BoxedIntEncoder)),
  outputEncoder = Some(value = StringEncoder),
  givenName = None,
  nullable = true,
  deterministic = true
)
res11_2: org.apache.spark.sql.expressions.UserDefinedFunction = SparkUserDefinedFunction(
  f = ammonite.$sess.cmd11$Helper$$Lambda$5876/0x0000024982508000@f0d48c1,
  dataType = StringType,
  inputEncoders = ArraySeq(Some(value = BoxedIntEncoder)),
  outputEncoder = Some(value = StringEncoder),
  givenName = Some(value = "categorizar_caso"),
  nullable = true,
  deterministic = true
)
tratamientosClasificados: org.apache.spark.sql.package.DataFrame = [id: int, tratamientos_texto: string ... 5 more fields]

---

## 🔧 Tarea 6 — Ranking de especialidades con `explode_outer`

**6a.** `explode_outer` conserva al paciente 7 (sin tratamientos): aparecerá con `especialidad = null`.
**6b.** Filtrar `null` y cadena vacía.
**6c.** Agrupar por `especialidad` y contar (los duplicados cuentan).


In [12]:
// 6a: explode_outer conserva todos los pacientes
val explotado = tratamientosArray
  .select($"id", explode_outer($"tratamientos_array").as("especialidad"))

println("=== explode_outer (incluye paciente 7 con null) ===")
explotado.show(truncate = false)

// 6b + 6c
val ranking = explotado
  .filter($"especialidad".isNotNull && $"especialidad" =!= "")
  .groupBy("especialidad")
  .count()
  .orderBy($"count".desc, $"especialidad")

println("=== Ranking de especialidades ===")
ranking.show(truncate = false)

=== explode_outer (incluye paciente 7 con null) ===
+---+--------------+
|id |especialidad  |
+---+--------------+
|1  |fisioterapia  |
|1  |radiología    |
|1  |análisis      |
|2  |cardiología   |
|2  |endocrinología|
|2  |fisioterapia  |
|2  |cardiología   |
|3  |análisis      |
|4  |traumatología |
|4  |fisioterapia  |
|4  |traumatología |
|4  |radiología    |
|5  |dermatología  |
|5  |análisis      |
|5  |dermatología  |
|6  |cardiología   |
|6  |radiología    |
|6  |neurología    |
|6  |cardiología   |
|6  |fisioterapia  |
+---+--------------+
only showing top 20 rows
=== Ranking de especialidades ===
+--------------+-----+
|especialidad  |count|
+--------------+-----+
|cardiología   |5    |
|fisioterapia  |4    |
|traumatología |4    |
|análisis      |3    |
|neurología    |3    |
|radiología    |3    |
|dermatología  |2    |
|endocrinología|1    |
+--------------+-----+



explotado: org.apache.spark.sql.package.DataFrame = [id: int, especialidad: string]
ranking: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [especialidad: string, count: bigint]

---

## 🔧 Tarea 7 — Informe final con Spark SQL

Registramos como vistas temporales el DataFrame de pacientes (Tarea 3) y el de tratamientos (Tarea 5), y ejecutamos un `LEFT JOIN` que conserva al paciente 7 aunque no tenga tratamientos.


In [13]:
pacientesPerfil.createOrReplaceTempView("pacientes")
tratamientosClasificados.createOrReplaceTempView("tratamientos")

val informeFinal = spark.sql("""
  SELECT
    p.id,
    p.nombre,
    p.edad,
    p.perfil_visitas,
    t.num_especialidades,
    categorizar_caso(t.num_especialidades) AS complejidad_caso,
    p.fecha_alta_formato_es
  FROM pacientes p
  LEFT JOIN tratamientos t ON p.id = t.id
  ORDER BY p.id
""")

informeFinal.show(truncate = false)

+---+-----------------+----+--------------+------------------+----------------+---------------------+
|id |nombre           |edad|perfil_visitas|num_especialidades|complejidad_caso|fecha_alta_formato_es|
+---+-----------------+----+--------------+------------------+----------------+---------------------+
|1  |Ana García Torres|40  |Ocasional     |3                 |Caso moderado   |01/03/2019           |
|2  |Pedro López      |53  |Habitual      |3                 |Caso moderado   |15/06/2021           |
|3  |María Ruiz       |36  |Nuevo         |1                 |Caso simple     |10/01/2023           |
|4  |Carlos Sanz      |57  |Habitual      |3                 |Caso moderado   |22/08/2020           |
|5  |Laura Vega       |25  |Ocasional     |2                 |Caso moderado   |01/12/2022           |
|6  |Jorge Martínez   |70  |Frecuente     |4                 |Caso complejo   |17/05/2018           |
|7  |Sofía Ramos      |30  |Ocasional     |NULL              |Sin datos       |28/

informeFinal: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string ... 5 more fields]

---

## ✅ Verificación final


In [14]:
println("=" * 55)
println("RESUMEN — Caso de Estudio SaludDirecta S.L.")
println("=" * 55)

val checks = Seq(
  ("Tarea 1 — pacientes limpios (8)",         pacientesLimpios.count() == 8),
  ("Tarea 1 — teléfonos solo dígitos",
    pacientesLimpios.filter($"telefono".rlike("^[0-9]+$")).count() == 8),
  ("Tarea 2 — columna 'edad' creada",         pacientesEnriquecidos.columns.contains("edad")),
  ("Tarea 3 — perfil_visitas asignado",       pacientesPerfil.filter($"perfil_visitas".isNotNull).count() == 8),
  ("Tarea 4 — paciente 7 excluido",           tratamientosAnalisis.count() == 7),
  ("Tarea 5 — complejidad asignada",          tratamientosClasificados.filter($"complejidad_caso".isNotNull).count() == 7),
  ("Tarea 6 — ranking 8 especialidades",      ranking.count() == 8),
  ("Tarea 7 — informe final con 8 filas",     informeFinal.count() == 8)
)

checks.foreach { case (desc, ok) =>
  println(s"${if (ok) "✅ CORRECTO" else "❌ REVISAR"} — $desc")
}

RESUMEN — Caso de Estudio SaludDirecta S.L.
✅ CORRECTO — Tarea 1 — pacientes limpios (8)
✅ CORRECTO — Tarea 1 — teléfonos solo dígitos
✅ CORRECTO — Tarea 2 — columna 'edad' creada
✅ CORRECTO — Tarea 3 — perfil_visitas asignado
✅ CORRECTO — Tarea 4 — paciente 7 excluido
✅ CORRECTO — Tarea 5 — complejidad asignada
✅ CORRECTO — Tarea 6 — ranking 8 especialidades
✅ CORRECTO — Tarea 7 — informe final con 8 filas


checks: Seq[(String, Boolean)] = List(
  ("Tarea 1 — pacientes limpios (8)", true),
  ("Tarea 1 — teléfonos solo dígitos", true),
  ("Tarea 2 — columna 'edad' creada", true),
  ("Tarea 3 — perfil_visitas asignado", true),
  ("Tarea 4 — paciente 7 excluido", true),
  ("Tarea 5 — complejidad asignada", true),
  ("Tarea 6 — ranking 8 especialidades", true),
  ("Tarea 7 — informe final con 8 filas", true)
)

## 🛑 Cierre

In [14]:
// spark.stop()